In [41]:
import os
if 'experiments' in os.getcwd():
    os.chdir(os.getcwd() + "/..")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore', category=ConvergenceWarning)

In [42]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet_pytorch.csv"

####### HYPERPARAMETERS #######
BATCH_SIZE = 32
LEARNING_RATE = 0.00003
NUM_EPOCHS = 200
HIDDEN_LAYERS = [100, 100, 100]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

Using device: cuda


# Neural Network Definition

In [43]:
class MilkYieldNet(nn.Module):
    def __init__(self, input_size, hidden_layers):
        super(MilkYieldNet, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.Tanh())
            prev_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(prev_size, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x).squeeze()

# Preprocessing

In [44]:
pd.set_option("display.max_columns", None)
TARGET_FEATURE = "Milk_Yield_L"

DROP_FEATURES = [
    "Cattle_ID",
    "Farm_ID",
    "Feed_Quantity_lb",
    "Breed",
    "Climate_Zone",
    "Management_System",
    "Feed_Type",
    "Feeding_Frequency",
    "Walking_Distance_km",
    "Grazing_Duration_hrs",
    "Rumination_Time_hrs",
    "Resting_Hours",
    "Body_Condition_Score",
    "Humidity_percent",
    "BVD_Vaccine",
    "FMD_Vaccine",
    "Brucellosis_Vaccine",
    "HS_Vaccine",
    "BQ_Vaccine",
    "Housing_Score",
]

CATEGORICAL_FEATURES = [
    "Lactation_Stage",
    "Date",
    "Milking_Interval_hrs",
]

STANDARD_SCALED_FEATURES = [
    "Age_Months",
    "Weight_kg",
    "Parity",
    "Days_in_Milk",
    "Feed_Quantity_kg",
    "Water_Intake_L",
    "Ambient_Temperature_C",
    "Previous_Week_Avg_Yield",
]

def preprocess(dtrain, dtest):
    # Convert month to season
    def month_to_season(m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    months = pd.to_datetime(dtest['Date']).dt.month
    dtest = dtest.drop(columns=['Date'])
    dtest['Date'] = months.apply(month_to_season)

    months = pd.to_datetime(dtrain['Date']).dt.month
    dtrain = dtrain.drop(columns=['Date'])
    dtrain['Date'] = months.apply(month_to_season)

    # Imputation
    median_val = dtrain["Feed_Quantity_kg"].median()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna(), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna(), "Feed_Quantity_kg"] = median_val

    # Drop features deemed unnecessary
    dtrain = dtrain.drop(DROP_FEATURES, axis=1)
    dtest = dtest.drop(DROP_FEATURES, axis=1)

    # One-hot encode
    dtrain = pd.get_dummies(dtrain, columns=CATEGORICAL_FEATURES, drop_first=True)
    dtest = pd.get_dummies(dtest, columns=CATEGORICAL_FEATURES, drop_first=True)
    dtrain, dtest = dtrain.align(dtest, join='left', axis=1, fill_value=0)

    # Standardize data
    scaler = StandardScaler()
    dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform(
                                                dtrain[STANDARD_SCALED_FEATURES])
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform(
                                                dtest[STANDARD_SCALED_FEATURES])

    dtrain = dtrain.replace({True: 1, False: 0})
    dtest = dtest.replace({True: 1, False: 0})


    return dtrain, dtest, scaler

# Training Functions

In [45]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_X.size(0)
    
    return total_loss / len(dataloader.dataset)


def evaluate(model, dataloader, device):
    model.eval()
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(batch_y.numpy())
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    
    return rmse, predictions, actuals

# Training Set Evaluation

In [46]:
train_data = pd.read_csv(TRAIN_PATH)
X_train, X_test, y_train, y_test = train_test_split(
    train_data.drop(TARGET_FEATURE, axis=1),
    train_data[TARGET_FEATURE], test_size=0.2, random_state=0)

X_train, X_test, scaler = preprocess(X_train, X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train.values)
y_train_tensor = torch.FloatTensor(y_train.values)
X_test_tensor = torch.FloatTensor(X_test.values)
y_test_tensor = torch.FloatTensor(y_test.values)

# Create data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

/tmp/ipykernel_63041/1857114196.py:86: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dtrain = dtrain.replace({True: 1, False: 0})
/tmp/ipykernel_63041/1857114196.py:87: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dtest = dtest.replace({True: 1, False: 0})


In [47]:
# Initialize model
input_size = X_train.shape[1]
model = MilkYieldNet(input_size, HIDDEN_LAYERS).to(DEVICE)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training loop
train_rmse_list = []
test_rmse_list = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    
    if epoch % 10 == 0 or epoch == 1:
        train_rmse, _, _ = evaluate(model, train_loader, DEVICE)
        test_rmse, _, _ = evaluate(model, test_loader, DEVICE)
        
        train_rmse_list.append(train_rmse)
        test_rmse_list.append(test_rmse)
        
        print(f"Epoch {epoch:3d}: Train RMSE={train_rmse:.4f}, Test RMSE={test_rmse:.4f}")

Epoch   1: Train RMSE=5.0148, Test RMSE=5.0387
Epoch  10: Train RMSE=4.1765, Test RMSE=4.1767
Epoch  20: Train RMSE=4.1646, Test RMSE=4.1662
Epoch  30: Train RMSE=4.1559, Test RMSE=4.1574
Epoch  40: Train RMSE=4.1396, Test RMSE=4.1423
Epoch  50: Train RMSE=4.1251, Test RMSE=4.1275
Epoch  60: Train RMSE=4.1205, Test RMSE=4.1225
Epoch  70: Train RMSE=4.1151, Test RMSE=4.1186
Epoch  80: Train RMSE=4.1133, Test RMSE=4.1184
Epoch  90: Train RMSE=4.1119, Test RMSE=4.1167
Epoch 100: Train RMSE=4.1067, Test RMSE=4.1132
Epoch 110: Train RMSE=4.1049, Test RMSE=4.1126
Epoch 120: Train RMSE=4.1032, Test RMSE=4.1119
Epoch 130: Train RMSE=4.1026, Test RMSE=4.1134
Epoch 140: Train RMSE=4.1007, Test RMSE=4.1123
Epoch 150: Train RMSE=4.0987, Test RMSE=4.1117
Epoch 160: Train RMSE=4.0975, Test RMSE=4.1123


KeyboardInterrupt: 

In [ ]:
# Final evaluation
train_rmse, train_pred, train_actual = evaluate(model, train_loader, DEVICE)
print(f"Train RMSE: {train_rmse:.4f}")

test_rmse, test_pred, test_actual = evaluate(model, test_loader, DEVICE)
print(f"Test RMSE: {test_rmse:.4f}")

# Analyze predictions
errors = np.sqrt((test_actual - test_pred) ** 2)
df_results = X_test.copy()
scaled_part = df_results[STANDARD_SCALED_FEATURES]
scaled_inverse = pd.DataFrame(scaler.inverse_transform(scaled_part),
                               columns=STANDARD_SCALED_FEATURES,
                               index=df_results.index)

df_results[STANDARD_SCALED_FEATURES] = scaled_inverse
df_results["y_true"] = test_actual
df_results["y_pred"] = test_pred
df_results["rmse"] = errors

raw_data = pd.read_csv(TRAIN_PATH)
df_results = df_results.merge(
    raw_data,
    left_index=True,
    right_index=True,
    how="left"
)

print("\nTop 5 BEST predictions:")
print(df_results.nsmallest(5, "rmse")[["y_true", "y_pred", "rmse"]])

print("\nTop 5 WORST predictions:")
print(df_results.nlargest(5, "rmse")[["y_true", "y_pred", "rmse"]])

# Final Model

In [ ]:
# Train final model on full dataset
train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

X_train_full = train_data.drop(TARGET_FEATURE, axis=1)
y_train_full = train_data[TARGET_FEATURE]
X_test_final = test_data

X_train_full, X_test_final, scaler_full = preprocess(X_train_full, X_test_final)

# Convert to tensors
X_train_full_tensor = torch.FloatTensor(X_train_full.values)
y_train_full_tensor = torch.FloatTensor(y_train_full.values)
X_test_final_tensor = torch.FloatTensor(X_test_final.values)

# Create data loader
full_train_dataset = TensorDataset(X_train_full_tensor, y_train_full_tensor)
full_train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize new model
final_model = MilkYieldNet(X_train_full.shape[1], HIDDEN_LAYERS).to(DEVICE)
final_optimizer = optim.Adam(final_model.parameters(), lr=LEARNING_RATE)

# Train final model
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(final_model, full_train_loader, criterion, final_optimizer, DEVICE)
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}: Loss={train_loss:.6f}")

In [ ]:
# Generate final predictions
final_model.eval()
with torch.no_grad():
    X_test_final_tensor = X_test_final_tensor.to(DEVICE)
    y_pred_final = final_model(X_test_final_tensor).cpu().numpy()

print(f"Mean predicted milk yield: {y_pred_final.mean():.4f} L")

# Save predictions
out_data = pd.DataFrame({
    'Cattle_ID': np.arange(1, len(y_pred_final) + 1),
    'Milk_Yield_L': y_pred_final
})
out_data.to_csv(OUT_PATH, index=False)

print(f"Predictions saved to {OUT_PATH}")